In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS earthquake_http_conn
TYPE HTTP
OPTIONS (
    host = "https://earthquake.usgs.gov",
    port = "443",
    base_path = "/earthquakes/feed/v1.0/",
    bearer_token = "NA"
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("earthquake_conn")
# print(conn)

base_path = f"{conn.options['host']}{conn.options['base_path']}"
print(base_path)

In [0]:
dbutils.widgets.text('catalog_name','earthquake_dev_catalog')
catalog_name = dbutils.widgets.get('catalog_name')
print(catalog_name)

dbutils.widgets.text('schema_name','bronze')
schema_name = dbutils.widgets.get('schema_name')
print(schema_name)


In [0]:
%py
spark.sql(f"USE CATALOG {catalog_name}");
spark.sql(f"USE SCHEMA {schema_name}");
spark.sql(f"CREATE VOLUME IF NOT EXISTS earthquake_volume");
# %sql
# USE CATALOG earthquake_dev_catalog;
# USE SCHEMA bronze;
# CREATE VOLUME IF NOT EXISTS earthquake_volume;

In [0]:
import requests
import json
from datetime import datetime

url = f"{base_path}/summary/all_day.geojson"
print(url)

response = requests.get(url)

data = response.json()
current_date = datetime.now().strftime("%Y-%m-%d")
print(current_date)
dbutils.fs.put(f"/Volumes/{catalog_name}/bronze/earthquake_volume/earthquake_data_{current_date}.json",
               json.dumps(data),
               overwrite=True)